# Voice Noise Analysis Round 1

这一份 notebook 用于完成第一轮分析：

- 读入 `clean / hiss / noisy` 三段音频
- 观察 waveform
- 观察 FFT magnitude spectrum
- 观察 spectrogram
- 试跑第一版 low-pass filter 并导出 `filtered_voice.wav`


In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from scipy import signal
from IPython.display import Audio, display

base = Path('/Users/xiejz959/Xiejz/College/Signal System & Probability/project/SSP_CP/Codes')
audio_dir = base / 'generated_audio'
chart_dir = Path('/Users/xiejz959/Xiejz/College/Signal System & Probability/project/SSP_CP/Charts/analysis_round1')
chart_dir.mkdir(parents=True, exist_ok=True)

clean, fs = sf.read(audio_dir / 'clean_voice.wav')
noise, _ = sf.read(audio_dir / 'hiss_noise.wav')
noisy, _ = sf.read(audio_dir / 'noisy_voice.wav')

print(f'fs = {fs}')
print(f'duration = {len(clean)/fs:.2f}s')


In [ ]:
print('Clean Voice')
display(Audio(clean, rate=fs))
print('Hiss Noise')
display(Audio(noise, rate=fs))
print('Noisy Voice')
display(Audio(noisy, rate=fs))


In [ ]:
t = np.arange(len(clean)) / fs

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
pairs = [
    ('Clean Voice', clean, 'tab:blue'),
    ('Hiss Noise', noise, 'tab:orange'),
    ('Noisy Voice', noisy, 'tab:red'),
]

for ax, (title, x, color) in zip(axes, pairs):
    ax.plot(t, x, color=color, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('Amp.')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()


In [ ]:
def magnitude_spectrum(x, fs):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    mag = np.abs(np.fft.rfft(x))
    return freqs, mag

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for ax, (title, x, color) in zip(axes, pairs):
    f, mag = magnitude_spectrum(x, fs)
    ax.plot(f, mag, color=color, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('Magnitude')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


In [ ]:
def plot_spectrogram(ax, x, fs, title):
    freqs, times, spec = signal.spectrogram(
        x,
        fs=fs,
        window='hann',
        nperseg=512,
        noverlap=384,
        scaling='spectrum',
        mode='magnitude',
    )
    spec_db = 20 * np.log10(spec + 1e-8)
    im = ax.pcolormesh(times, freqs, spec_db, shading='gouraud', cmap='magma')
    ax.set_title(title)
    ax.set_ylabel('Frequency (Hz)')
    return im

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
im = plot_spectrogram(axes[0], clean, fs, 'Clean Voice')
plot_spectrogram(axes[1], noise, fs, 'Hiss Noise')
plot_spectrogram(axes[2], noisy, fs, 'Noisy Voice')
axes[-1].set_xlabel('Time (s)')
fig.colorbar(im, ax=axes, format='%+2.0f dB')
plt.tight_layout()
plt.show()


In [ ]:
def normalize_peak(x, peak=0.92):
    m = np.max(np.abs(x))
    if m == 0:
        return x.copy()
    return x * (peak / m)

def lowpass_filter(x, fs, cutoff, order=4):
    b, a = signal.butter(order, cutoff, btype='low', fs=fs)
    y = signal.filtfilt(b, a, x)
    return normalize_peak(y)

def snr_db(reference, estimate):
    err = estimate - reference
    return 10 * np.log10(np.sum(reference**2) / np.sum(err**2))

input_snr = snr_db(clean, noisy)
print('Input SNR:', round(float(input_snr), 3), 'dB')

for cutoff in [2800, 3200, 3600]:
    filt = lowpass_filter(noisy, fs, cutoff, order=4)
    out_snr = snr_db(clean, filt)
    corr = np.corrcoef(clean, filt)[0, 1]
    print(cutoff, 'Hz -> output SNR =', round(float(out_snr), 3), 'dB, corr =', round(float(corr), 4))


In [ ]:
selected_cutoff = 3200
filtered = lowpass_filter(noisy, fs, selected_cutoff, order=4)
sf.write(audio_dir / 'filtered_voice.wav', filtered, fs)

print('Filtered Voice')
display(Audio(filtered, rate=fs))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(t, noisy, color='tab:red', linewidth=0.8)
axes[0].set_title('Noisy Voice')
axes[0].set_ylabel('Amp.')
axes[0].grid(alpha=0.25)

axes[1].plot(t, filtered, color='tab:green', linewidth=0.8)
axes[1].set_title('Filtered Voice')
axes[1].set_ylabel('Amp.')
axes[1].set_xlabel('Time (s)')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()
